<a href="https://colab.research.google.com/github/sakshama1307/catsvsdogs/blob/main/bck.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential,Input
from keras.layers import Conv2D,MaxPooling2D,Dense,Flatten,Rescaling,Input

In [ ]:
train_ds=keras.utils.image_dataset_from_directory(
    directory='/content/train/catsvsdogs/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256)
)

Found 19999 files belonging to 2 classes.


In [ ]:
test_ds=keras.utils.image_dataset_from_directory(
    directory='/content/train/catsvsdogs/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256,256))

Found 5000 files belonging to 2 classes.


In [ ]:
model = keras.Sequential([
    Input(shape=(256, 256, 3)),
    # Rescale pixel values from [0, 255] to [0, 1]
    Rescaling(1./255),

    # Your layers follow here
    Conv2D(32, (3, 3), activation='relu',padding='valid'),
    MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'),

    Conv2D(32, (3, 3), activation='relu',padding='valid'),
    MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'),

    Conv2D(32, (3, 3), activation='relu',padding='valid'),
    MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'),

    Flatten(),
    Dense(128, activation='relu'),

    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 125, 125, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 60, 60, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 28800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │     3,686,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,714,241 (14.17 MB)

 Trainable params: 3,714,241 (14.17 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# 1. Early Stopping: Stop training if val_loss doesn't improve for 5 epochs
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# 2. Model Checkpoint: Save the best version of your model to disk
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# 3. Reduce LR on Plateau: Lower learning rate if training plateaus
rlrp = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

# Combine them into a list
callbacks_list = [early_stopping, checkpoint, rlrp]

In [ ]:
print(tf.__version__)

2.20.0


In [ ]:
history=model.fit(train_ds,epochs=10,validation_data=test_ds,callbacks=callbacks_list)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5342 - loss: 0.6926
Epoch 1: val_loss improved from None to 0.58147, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 69ms/step - accuracy: 0.5836 - loss: 0.6611 - val_accuracy: 0.6912 - val_loss: 0.5815 - learning_rate: 0.0010
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.7279 - loss: 0.5368
Epoch 2: val_loss improved from 0.58147 to 0.46297, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 58ms/step - accuracy: 0.7493 - loss: 0.5074 - val_accuracy: 0.7778 - val_loss: 0.4630 - learning_rate: 0.0010
Epoch 3/10
624/625 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.7924 - loss: 0.4383
Epoch 3: val_loss did not improve from 0.46297
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 58ms/step - accuracy: 0.8068 - loss: 0.4172 - val_accuracy: 0.7852 - val_loss: 0.4635 - learning_rate: